# GenAI-Traces: Fine-Tuning Dataset Export

This notebook demonstrates how to export high-quality production traces as labeled datasets for fine-tuning LLMs.

## Features Covered
1. Basic export from spans to JSONL
2. Multiple output formats (OpenAI, HuggingFace, Alpaca, ShareGPT)
3. Quality filtering based on eval scores and feedback
4. Deduplication by prompt hash
5. Custom filtering functions
6. Dataset validation
7. Train/val/test splitting
8. Dataset merging

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
from pathlib import Path

from genai_traces.exporters import FineTuneExporter, FineTuneRecord, DatasetFormat

# Create output directory for this notebook
OUTPUT_DIR = Path("./finetune_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Fine-Tuning Export module loaded!")

## 1. Create Sample Span Data

First, let's create some sample span data that simulates production traces.

In [ ]:
# Sample spans simulating production LLM traces
sample_spans = [
    {
        "trace_id": "trace_001",
        "span_id": "span_001",
        "name": "chat_completion",
        "start_time": "2024-01-15T10:00:00Z",
        "attributes": {
            "llm.prompt": "What is the capital of France?",
            "llm.completion": "The capital of France is Paris. Paris is located in northern France and is the country's largest city.",
            "llm.model.name": "gpt-4o",
            "llm.prompt.tokens": 8,
            "llm.completion.tokens": 25,
            "llm.total_tokens": 33,
            "eval.quality": 0.95,
            "eval.relevance": 0.98,
            "feedback.score": 5,
            "feedback.comment": "Perfect answer!",
            "cost.total_usd": 0.001,
            "llm.duration_ms": 450
        }
    },
    {
        "trace_id": "trace_002",
        "span_id": "span_002",
        "name": "chat_completion",
        "start_time": "2024-01-15T10:05:00Z",
        "attributes": {
            "llm.prompt": "Explain quantum computing in simple terms.",
            "llm.completion": "Quantum computing uses quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits. This allows quantum computers to process many calculations at once.",
            "llm.model.name": "gpt-4o",
            "llm.prompt.tokens": 7,
            "llm.completion.tokens": 40,
            "eval.quality": 0.88,
            "feedback.score": 4
        }
    },
    {
        "trace_id": "trace_003",
        "span_id": "span_003",
        "name": "chat_completion",
        "start_time": "2024-01-15T10:10:00Z",
        "attributes": {
            "llm.prompt": "Write a haiku about programming.",
            "llm.completion": "Code flows like water\nBugs emerge from the shadows\nDebug, compile, run",
            "llm.model.name": "gpt-4o",
            "eval.quality": 0.92,
            "feedback.score": 5
        }
    },
    {
        "trace_id": "trace_004",
        "span_id": "span_004",
        "name": "chat_completion",
        "start_time": "2024-01-15T10:15:00Z",
        "attributes": {
            "llm.prompt": "What is 2+2?",
            "llm.completion": "2+2 equals 4.",
            "llm.model.name": "gpt-3.5-turbo",
            "eval.quality": 0.50,  # Low quality - will be filtered
            "feedback.score": 2
        }
    },
    {
        "trace_id": "trace_005",
        "span_id": "span_005",
        "name": "chat_completion",
        "start_time": "2024-01-15T10:20:00Z",
        "attributes": {
            "llm.system_prompt": "You are a helpful coding assistant.",
            "llm.messages": [
                {"role": "system", "content": "You are a helpful coding assistant."},
                {"role": "user", "content": "How do I read a file in Python?"},
                {"role": "assistant", "content": "Use the open() function with a context manager: with open('file.txt', 'r') as f: content = f.read()"}
            ],
            "llm.prompt": "How do I read a file in Python?",
            "llm.completion": "Use the open() function with a context manager: with open('file.txt', 'r') as f: content = f.read()",
            "llm.model.name": "gpt-4o",
            "eval.quality": 0.90,
            "feedback.score": 5
        }
    },
    {
        "trace_id": "trace_006",
        "span_id": "span_006",
        "name": "chat_completion",
        "start_time": "2024-01-15T10:25:00Z",
        "attributes": {
            "llm.prompt": "What is the capital of France?",  # Duplicate prompt
            "llm.completion": "Paris is the capital of France.",
            "llm.model.name": "gpt-4o",
            "eval.quality": 0.85,
            "feedback.score": 4
        }
    },
    {
        "trace_id": "trace_007",
        "span_id": "span_007",
        "name": "chat_completion",
        "start_time": "2024-01-15T10:30:00Z",
        "attributes": {
            # Missing completion - will be filtered
            "llm.prompt": "Tell me a joke.",
            "llm.model.name": "gpt-4o",
            "eval.quality": 0.80
        }
    }
]

print(f"Created {len(sample_spans)} sample spans")

## 2. Basic Export to OpenAI Format

Export spans to OpenAI's fine-tuning JSONL format.

In [ ]:
# Create exporter with default settings
exporter = FineTuneExporter(
    min_quality_score=0.7,
    min_feedback_score=4,
    dedup=True,
    format=DatasetFormat.OPENAI
)

# Export to file
output_path = OUTPUT_DIR / "openai_dataset.jsonl"
count = exporter.export_from_spans(sample_spans, str(output_path))

print(f"Exported {count} records to {output_path}")
print(f"\nExport statistics:")
for key, value in exporter.get_stats().items():
    print(f"  {key}: {value}")

# Show sample output
print(f"\nSample output (first 2 records):")
with open(output_path) as f:
    for i, line in enumerate(f):
        if i >= 2:
            break
        print(json.dumps(json.loads(line), indent=2))

## 3. Export to Different Formats

Export the same data to HuggingFace, Alpaca, and ShareGPT formats.

In [ ]:
# HuggingFace format
hf_exporter = FineTuneExporter(
    min_quality_score=0.7,
    format=DatasetFormat.HUGGINGFACE
)
hf_path = OUTPUT_DIR / "hf_dataset.jsonl"
hf_count = hf_exporter.export_from_spans(sample_spans, str(hf_path))

print("HuggingFace Format:")
with open(hf_path) as f:
    print(json.dumps(json.loads(f.readline()), indent=2))

# Alpaca format
alpaca_exporter = FineTuneExporter(
    min_quality_score=0.7,
    format=DatasetFormat.ALPACA
)
alpaca_path = OUTPUT_DIR / "alpaca_dataset.jsonl"
alpaca_count = alpaca_exporter.export_from_spans(sample_spans, str(alpaca_path))

print("\nAlpaca Format:")
with open(alpaca_path) as f:
    print(json.dumps(json.loads(f.readline()), indent=2))

# ShareGPT format
sharegpt_exporter = FineTuneExporter(
    min_quality_score=0.7,
    format=DatasetFormat.SHAREGPT
)
sharegpt_path = OUTPUT_DIR / "sharegpt_dataset.jsonl"
sharegpt_count = sharegpt_exporter.export_from_spans(sample_spans, str(sharegpt_path))

print("\nShareGPT Format:")
with open(sharegpt_path) as f:
    print(json.dumps(json.loads(f.readline()), indent=2))

## 4. Custom Filtering

Use custom filter functions to select specific spans for export.

In [ ]:
# Filter for only GPT-4o model responses
def gpt4o_only(span):
    model = span.get("attributes", {}).get("llm.model.name", "")
    return "gpt-4o" in model.lower()

gpt4_exporter = FineTuneExporter(
    min_quality_score=0.7,
    filter_fn=gpt4o_only,
    format=DatasetFormat.OPENAI
)

gpt4_path = OUTPUT_DIR / "gpt4o_only.jsonl"
gpt4_count = gpt4_exporter.export_from_spans(sample_spans, str(gpt4_path))

print(f"Exported {gpt4_count} GPT-4o records")
print(f"\nStats: {gpt4_exporter.get_stats()}")

In [ ]:
# Filter for coding-related prompts
def coding_filter(span):
    prompt = span.get("attributes", {}).get("llm.prompt", "").lower()
    coding_keywords = ["code", "python", "function", "program", "file", "debug"]
    return any(keyword in prompt for keyword in coding_keywords)

coding_exporter = FineTuneExporter(
    min_quality_score=0.7,
    filter_fn=coding_filter,
    format=DatasetFormat.OPENAI
)

coding_path = OUTPUT_DIR / "coding_dataset.jsonl"
coding_count = coding_exporter.export_from_spans(sample_spans, str(coding_path))

print(f"Exported {coding_count} coding-related records")

## 5. Include Metadata

Export with metadata for tracking and analysis.

In [ ]:
metadata_exporter = FineTuneExporter(
    min_quality_score=0.7,
    format=DatasetFormat.OPENAI,
    include_metadata=True
)

metadata_path = OUTPUT_DIR / "with_metadata.jsonl"
metadata_count = metadata_exporter.export_from_spans(sample_spans, str(metadata_path))

print("Export with metadata:")
with open(metadata_path) as f:
    record = json.loads(f.readline())
    print(json.dumps(record, indent=2))

## 6. FineTuneRecord Class

Explore the `FineTuneRecord` dataclass for manual record creation.

In [ ]:
# Create a FineTuneRecord manually
record = FineTuneRecord(
    prompt="Explain the difference between a list and a tuple in Python.",
    completion="Lists are mutable (can be modified), while tuples are immutable (cannot be changed after creation). Lists use square brackets [], tuples use parentheses ().",
    quality=0.95,
    source_trace_id="manual_001",
    source_span_id="span_manual_001",
    system_prompt="You are a Python expert.",
    model="gpt-4o",
    feedback_score=5,
    prompt_tokens=15,
    completion_tokens=35
)

print("OpenAI format:")
print(json.dumps(record.to_openai_format(), indent=2))

print("\nHuggingFace format:")
print(json.dumps(record.to_hf_format(), indent=2))

print("\nAlpaca format:")
print(json.dumps(record.to_alpaca_format(), indent=2))

print("\nShareGPT format:")
print(json.dumps(record.to_sharegpt_format(), indent=2))

## 7. Dataset Validation

Validate exported datasets for quality and completeness.

In [ ]:
# Validate the OpenAI dataset
validation_result = exporter.validate_dataset(str(output_path))

print("Dataset Validation Results:")
print(f"  Valid: {validation_result['valid']}")
print(f"  Record count: {validation_result['record_count']}")
print(f"  Empty prompts: {validation_result['empty_prompts']}")
print(f"  Empty completions: {validation_result['empty_completions']}")
print(f"  Avg prompt words: {validation_result['avg_prompt_words']:.1f}")
print(f"  Avg completion words: {validation_result['avg_completion_words']:.1f}")

if validation_result['issues']:
    print(f"  Issues: {validation_result['issues']}")

## 8. Train/Val/Test Split

Split datasets for model training and evaluation.

In [ ]:
# Create a larger dataset for splitting
large_spans = []
for i in range(100):
    large_spans.append({
        "trace_id": f"trace_{i:03d}",
        "span_id": f"span_{i:03d}",
        "attributes": {
            "llm.prompt": f"Question {i}: What is {i} + {i}?",
            "llm.completion": f"The answer is {i + i}.",
            "llm.model.name": "gpt-4o",
            "eval.quality": 0.8 + (i % 20) * 0.01
        }
    })

# Export large dataset
large_exporter = FineTuneExporter(min_quality_score=0.7, dedup=False)
large_path = OUTPUT_DIR / "large_dataset.jsonl"
large_exporter.export_from_spans(large_spans, str(large_path))

# Split into train/val/test
split_dir = OUTPUT_DIR / "split"
split_paths = FineTuneExporter.split_dataset(
    str(large_path),
    str(split_dir),
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    shuffle=True,
    seed=42
)

print("Dataset split:")
for split_name, split_path in split_paths.items():
    with open(split_path) as f:
        count = sum(1 for _ in f)
    print(f"  {split_name}: {count} records ({split_path})")

## 9. Dataset Merging

Merge multiple datasets into one.

In [ ]:
# Merge the OpenAI and coding datasets
merged_path = OUTPUT_DIR / "merged_dataset.jsonl"
merged_count = FineTuneExporter.merge_datasets(
    [str(output_path), str(coding_path)],
    str(merged_path),
    dedup=True
)

print(f"Merged {merged_count} unique records into {merged_path}")

## 10. Export from JSONL File

Export from an existing JSONL file of spans.

In [ ]:
# First, create a JSONL file of spans
spans_file = OUTPUT_DIR / "raw_spans.jsonl"
with open(spans_file, 'w') as f:
    for span in sample_spans:
        f.write(json.dumps(span) + "\n")

print(f"Created spans file: {spans_file}")

# Export from the JSONL file
file_exporter = FineTuneExporter(
    min_quality_score=0.7,
    format=DatasetFormat.OPENAI
)

from_file_path = OUTPUT_DIR / "from_file_dataset.jsonl"
from_file_count = file_exporter.export_from_jsonl(
    str(spans_file),
    str(from_file_path)
)

print(f"Exported {from_file_count} records from JSONL file")

## 11. Compressed Export

Export with gzip compression for large datasets.

In [ ]:
import gzip

compressed_exporter = FineTuneExporter(
    min_quality_score=0.7,
    format=DatasetFormat.OPENAI,
    compress=True
)

compressed_path = OUTPUT_DIR / "compressed_dataset.jsonl.gz"
compressed_count = compressed_exporter.export_from_spans(sample_spans, str(compressed_path))

# Check file sizes
uncompressed_size = os.path.getsize(output_path)
compressed_size = os.path.getsize(compressed_path)

print(f"Exported {compressed_count} records (compressed)")
print(f"Uncompressed size: {uncompressed_size} bytes")
print(f"Compressed size: {compressed_size} bytes")
print(f"Compression ratio: {uncompressed_size / compressed_size:.1f}x")

# Read compressed file
print("\nFirst record from compressed file:")
with gzip.open(compressed_path, 'rt') as f:
    print(json.dumps(json.loads(f.readline()), indent=2))

## 12. Quality-Based Filtering Examples

Demonstrate different quality thresholds.

In [ ]:
# Compare different quality thresholds
thresholds = [0.5, 0.7, 0.8, 0.9]

print("Records exported at different quality thresholds:")
print("-" * 50)

for threshold in thresholds:
    exp = FineTuneExporter(
        min_quality_score=threshold,
        min_feedback_score=1,  # Don't filter by feedback
        dedup=False
    )
    path = OUTPUT_DIR / f"quality_{int(threshold*100)}.jsonl"
    count = exp.export_from_spans(sample_spans, str(path))
    print(f"  Quality >= {threshold}: {count} records")

## 13. Multi-Turn Conversation Export

Export conversations with multiple turns.

In [ ]:
# Multi-turn conversation span
conversation_span = {
    "trace_id": "conv_001",
    "span_id": "span_conv_001",
    "attributes": {
        "llm.system_prompt": "You are a helpful math tutor.",
        "llm.messages": [
            {"role": "system", "content": "You are a helpful math tutor."},
            {"role": "user", "content": "What is calculus?"},
            {"role": "assistant", "content": "Calculus is a branch of mathematics that studies continuous change."},
            {"role": "user", "content": "Can you give me an example?"},
            {"role": "assistant", "content": "Sure! Finding the slope of a curve at any point is a calculus problem. For example, if you have y = x^2, calculus tells us the slope at any point x is 2x."}
        ],
        "llm.prompt": "Can you give me an example?",
        "llm.completion": "Sure! Finding the slope of a curve at any point is a calculus problem. For example, if you have y = x^2, calculus tells us the slope at any point x is 2x.",
        "llm.model.name": "gpt-4o",
        "eval.quality": 0.95
    }
}

# Export conversation
conv_exporter = FineTuneExporter(
    min_quality_score=0.7,
    format=DatasetFormat.OPENAI
)

conv_path = OUTPUT_DIR / "conversation_dataset.jsonl"
conv_exporter.export_from_spans([conversation_span], str(conv_path))

print("Multi-turn conversation in OpenAI format:")
with open(conv_path) as f:
    print(json.dumps(json.loads(f.readline()), indent=2))

## Cleanup

In [ ]:
# Optional: Clean up output files
import shutil

cleanup = False  # Set to True to delete output files

if cleanup:
    shutil.rmtree(OUTPUT_DIR)
    print(f"Cleaned up {OUTPUT_DIR}")
else:
    print(f"Output files preserved in {OUTPUT_DIR}")
    print("\nFiles created:")
    for f in OUTPUT_DIR.rglob("*"):
        if f.is_file():
            print(f"  {f.relative_to(OUTPUT_DIR)}")

## Summary

This notebook demonstrated the fine-tuning dataset export capabilities:

1. **Multiple Formats**: Export to OpenAI, HuggingFace, Alpaca, and ShareGPT formats
2. **Quality Filtering**: Filter by quality scores and feedback ratings
3. **Deduplication**: Remove duplicate prompts automatically
4. **Custom Filtering**: Apply custom filter functions for specific use cases
5. **Metadata**: Include source trace information for tracking
6. **Validation**: Validate exported datasets for completeness
7. **Splitting**: Create train/val/test splits for model training
8. **Merging**: Combine multiple datasets
9. **Compression**: Gzip compression for large datasets
10. **Multi-turn**: Support for conversation history

These features enable you to create high-quality fine-tuning datasets from your production LLM traces.